# `parse_pdf` — analyse complète d'un PDF en 1 ouverture fitz

Module testé : [`src/docpipeline/parsing/pdf/parse_pdf.py`](../src/docpipeline/parsing/pdf/parse_pdf.py).

Le script ouvre fitz **une seule fois** et retourne 4 sorties prêtes à être consommées en aval :

| Sortie | Schéma |
|---|---|
| `line_df`  | 1 ligne = 1 ligne de texte (texte, bbox, font, taille, `is_invisible`) |
| `image_df` | 1 ligne = 1 image (bbox affichée en points PDF, dimensions, ratio de couverture) |
| `page_df`  | 1 ligne = 1 page (`page_type`, flags additifs, `extraction_strategy`, `tool`) |
| `doc_summary` | JSON synthèse : `source_tool`, `content_type`, `recommended_strategy`, `pages_needing_ocr` |

**page_type** (mutuellement exclusif) : `native`, `native_with_image`, `scanned`, `scanned_ocr_good`, `scanned_ocr_bad`, `mixed`, `empty`, `unknown`.

**extraction_strategy** : `native` (fitz direct), `ocr` (OCR obligatoire), `hybrid` (texte natif + OCR sur images), `skip` (page vide).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 1 — Démonstration sur un PDF du corpus client

Cas typique : un contrat MRH InDesign avec couverture scannée. On affiche le `doc_summary` complet + le `page_df`.

In [2]:
import json
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

PDF = '../data/CG contrats MRH/CG Assistance aux personnes AXA.pdf'
result = parse_pdf(PDF)

print('=== doc_summary ===')
print(json.dumps(result['doc_summary'], indent=2, ensure_ascii=False, default=str))
print()
print('=== page_df ===')
print(result['page_df'][['page_num', 'page_type', 'tool', 'extraction_strategy', 'char_count', 'n_images', 'image_coverage_ratio']].to_string(index=False))
print()
print(f"line_df  : {len(result['line_df'])} lignes")
print(f"image_df : {len(result['image_df'])} images")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


=== doc_summary ===
{
  "pdf_hash": "4bb8af750f0f3ad6fe815d15a3830016f9a6c42d9d9576a23ac437eed9b95014",
  "file_size_bytes": 239954,
  "n_pages": 20,
  "source_category": "design_tool",
  "source_tool": "Adobe InDesign",
  "source_confidence": 0.95,
  "source_signals": [
    "creator~adobe indesign",
    "xmp_history~adobe indesign"
  ],
  "creator_raw": "Adobe InDesign 14.0 (Macintosh)",
  "producer_raw": "Adobe PDF Library 15.0",
  "pdf_version": "1.6",
  "creation_date": "D:20210308154957+01'00'",
  "modification_date": "D:20210521105223+02'00'",
  "content_type": "mixed",
  "is_scanned": false,
  "has_text_layer": true,
  "ocr_quality": "not_applicable",
  "ocr_quality_score": null,
  "page_type_counts": {
    "unknown": 1,
    "native": 16,
    "empty": 2,
    "native_with_image": 1
  },
  "scanned_page_ratio": 0.0,
  "native_page_ratio": 0.944,
  "is_encrypted": false,
  "has_form_fields": false,
  "n_vector_tables": 0,
  "n_images_total": 12,
  "recommended_strategy": "per_page_

### Lecture des résultats

Sur `CG Assistance aux personnes AXA.pdf` (20 pages) :
- `source_tool: Adobe InDesign` détecté via les metadata du fichier.
- `content_type: mixed` → le PDF n'est pas 100 % natif : la page de couverture est `scanned_ocr_good` (image plein format avec couche OCR exploitable), les pages 1-19 sont `native_with_image` (texte natif + logos/illustrations).
- `recommended_strategy: per_page_routing` → le pipeline aval doit router page par page (page 0 → réutiliser l'OCR existant ; pages 1-19 → fitz direct).
- `pages_needing_ocr: []` → la couche OCR de la page 0 est exploitable, pas besoin de ré-OCR.
- Les colonnes `extraction_strategy` du `page_df` confirment : toutes en `native` (la page 0 utilise sa couche OCR existante).

## 2 — Banque de cas (test visuel sur le corpus client)

Couvre les profils typiques rencontrés sur le corpus : contrats MRH InDesign, rapports de solvabilité avec creator inconnu, doc avec page scannée incrustée, exports recompressés Ghostscript, exports Word, papers académiques.

In [3]:
from pathlib import Path
import pandas as pd
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

REPO_ROOT = Path('..').resolve()

CASES = [
    ('data/CG contrats MRH/CG allianz MRH.pdf',                 'InDesign / native'),
    ('data/CG contrats MRH/CG Habitation_MMA_410s.pdf',         'InDesign / native'),
    ('data/CG contrats MRH/CG Assistance aux personnes AXA.pdf','InDesign / mixed'),
    ('data/annuel_reports/ABEILLE ASSURANCES/09 SFCR 2022 - 10 Abeille Vie.pdf', 'Unknown / native'),
    ('data/cmo/CMO-April-2024.pdf',                             'Ghostscript / native'),
    ('data/insurance/agcs euroapi.pdf',                         'Microsoft Word / native'),
    ('data/paper/1706.03762v7.pdf',                             'Unknown / native (Attention is all you need)'),
    ('data/reports/sdg_7_2025.pdf',                             'InDesign / mixed (gros doc)'),
]

rows = []
for rel, expected in CASES:
    pdf = REPO_ROOT / rel
    if not pdf.exists():
        rows.append({'file': Path(rel).name, 'expected': expected, 'status': '(missing)'})
        continue
    s = parse_pdf(pdf)['doc_summary']
    rows.append({
        'file':          Path(rel).name,
        'expected':      expected,
        'n_pages':       s['n_pages'],
        'source_tool':   s['source_tool'],
        'content_type':  s['content_type'],
        'reco':          s['recommended_strategy'],
        'pages_OCR':     len(s['pages_needing_ocr']),
    })

print(pd.DataFrame(rows).to_string(index=False))

                               file                                     expected  n_pages    source_tool content_type                  reco  pages_OCR
                 CG allianz MRH.pdf                            InDesign / native      104 Adobe InDesign       native use_native_extraction          0
         CG Habitation_MMA_410s.pdf                            InDesign / native       80 Adobe InDesign       native use_native_extraction          0
CG Assistance aux personnes AXA.pdf                             InDesign / mixed       20 Adobe InDesign        mixed      per_page_routing          0
  09 SFCR 2022 - 10 Abeille Vie.pdf                             Unknown / native       52        Unknown       native use_native_extraction          0
                 CMO-April-2024.pdf                         Ghostscript / native       52    Ghostscript       native use_native_extraction          0
                   agcs euroapi.pdf                      Microsoft Word / native       77 Micr

### Lecture des résultats

Pour chaque cas, la colonne `expected` (profil attendu) doit matcher avec `source_tool` + `content_type` :
- **MRH Allianz / MMA** : InDesign / native → ✅ contrats statiques, extractibles directement.
- **MRH AXA Assistance** : InDesign / mixed → ✅ couverture incrustée, routage page-par-page.
- **SFCR Abeille** : creator absent dans les metadata (Unknown) mais contenu native → extraction directe possible.
- **CMO** : Ghostscript = recompression intermédiaire qui a écrasé le creator d'origine, mais le contenu reste native.
- **Paper académique** : creator absent (typique des exports LaTeX/scientific) mais native pur.
- **sdg_7_2025** (188 pages) : InDesign / mixed → quelques pages avec image de couverture, le reste native ; `pages_OCR` indique combien de pages exigent un OCR.

Aucun PDF du corpus ne sort en `source_tool: scanner` → tout le corpus est éditorial / numérique, pas de scanned-only.

## 3 — Bench complet sur les 71 PDFs

Pour le bench complet, voir `bench_parse_pdf_js.ipynb` dans le même dossier. Sortie consolidée : `bench_parse_pdf_results_js.csv` à la racine du repo (71 PDFs traités, 6 589 pages, 0 erreur, 43 min).